# Step 10: Real-World Application (Seafood and Sustainability Claims)

### Explanation of the notebook:

Runs 30 real, manually labelled social-media claims about seafood and sustainability through the
project's pipeline against SciFact-Open, under three conditions: no retrieval (Model 1), plain
dense retrieval (Model 2), and dense plus soft stance reranking (Model 2), with the two retrieval
conditions swept over k = 1, 3, 5, 10. This is the domain-generalisation test: everything earlier
in the project is biomedical, so this step asks whether the findings transfer to a different
subject domain. Poor retrieval here is expected and, if observed, is part of the phenomenon being
studied rather than a bug, because a biomedical corpus mostly has no evidence for seafood claims.

The run should save 270 per-claim records (30 no-retrieval + 30x4 dense + 30x4 reranked). The
analysis is qualitative, following results_annotation_guide.md, and is organised around five
pre-committed hypotheses (H1 to H5) carried in from Steps 4, 7, 8 and 9.

In [1]:
#mounting Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/rag-thesis/results', exist_ok=True)
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


In [2]:
#checking the GPU

'''
This step encodes the roughly 500k SciFact-Open corpus once.
'''

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only - STOP, attach a GPU")

CUDA available: True
Device: NVIDIA A100-SXM4-40GB


In [3]:
#cloning the repo on the Step 10 branch
import os

#removing any previous clone to start clean
if os.path.exists('/content/retrieval-effects-claim-verification'):
    import shutil
    shutil.rmtree('/content/retrieval-effects-claim-verification')

#cloning the private repo using the GitHub token for authentication
!git clone https://github.com/pernillejorg/retrieval-effects-claim-verification.git

%cd retrieval-effects-claim-verification

!git checkout step10-realworld
!git pull origin step10-realworld

print("Repository cloned and on step10-realworld branch.")

Cloning into 'rag-claim-verification'...
remote: Enumerating objects: 843, done.
remote: Counting objects: 100% (291/291), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 843 (delta 147), reused 243 (delta 107), pack-reused 552 (from 1)
Receiving objects: 100% (843/843), 13.39 MiB | 18.21 MiB/s, done.
Resolving deltas: 100% (467/467), done.
/content/retrieval-effects-claim-verification
Branch 'step10-realworld' set up to track remote branch 'step10-realworld' from 'origin'.
Switched to a new branch 'step10-realworld'
From https://github.com/pernillejorg/rag-claim-verification
 * branch            step10-realworld -> FETCH_HEAD
Already up to date.
Repository cloned and on step10-realworld branch.


In [4]:
#installing dependencies

'''
The case-study script itself imports the project pipeline, which uses transformers,
sentence-transformers and the SciFact-Open loader, so the same dependencies as the earlier steps
are installed here.
'''

!pip install "datasets==2.21.0" -q
!pip install -r requirements.txt -q
import datasets, transformers, sentence_transformers
print("datasets", datasets.__version__, "| transformers", transformers.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 22.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.6/120.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 132.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 132.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 133.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 61.4 MB/s eta 0:00:00
   ━━━━━━━

In [5]:
#staging the two trained models and the SciFact-Open cache

'''
Step 10 needs the two Step 2 classifiers (Model 1 claim-only for no retrieval, Model 2
claim+evidence for the retrieval conditions), and the SciFact-Open corpus. The models are copied
from Drive; the corpus cache is staged so the 500k documents load from disk rather than being
re-downloaded. The seafood claim set should already be in the repo under realworld/.
'''

import os, shutil
REPO = '/content/retrieval-effects-claim-verification'
DRIVE = '/content/drive/MyDrive/rag-thesis'

os.makedirs(f'{REPO}/models/saved_models', exist_ok=True)
for m in ['baseline_scifact', 'evidence_scifact']:
    src, dst = f'{DRIVE}/models/saved_models/{m}', f'{REPO}/models/saved_models/{m}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("staged model:", m)
    else:
        print("WARNING: model not found in Drive:", src)

src_cache = f'{DRIVE}/data/scifact_open/cache'
dst_cache = f'{REPO}/data/scifact_open/cache'
if os.path.exists(src_cache):
    os.makedirs(dst_cache, exist_ok=True)
    shutil.copytree(src_cache, dst_cache, dirs_exist_ok=True)
    print("staged SciFact-Open cache")
else:
    print("no SciFact-Open cache in Drive; it will be downloaded on first load")

print("claims CSV present:", os.path.exists(f'{REPO}/realworld/seafood_claims.csv'))

staged model: baseline_scifact
staged model: evidence_scifact
staged SciFact-Open cache
claims CSV present: True


In [6]:
#confirming the claim set is clean and balanced before the long run

'''
Just performing a quick check before the expensive run: 30 claims, valid labels, unique ids, and the near/far and
label balance that the analysis relies on.
'''

import csv
from collections import Counter
rows = list(csv.DictReader(open('realworld/seafood_claims.csv')))
print("claims:", len(rows))
print("labels:", dict(Counter(r['true_label'] for r in rows)))
print("corpus_fit:", dict(Counter(r['corpus_fit'] for r in rows)))
assert all(r['true_label'] in {'SUPPORT','CONTRADICT','NEI'} for r in rows), "bad label"
assert len({r['claim_id'] for r in rows}) == len(rows), "duplicate ids"
print("OK - claim set is clean")

claims: 30
labels: {'CONTRADICT': 7, 'SUPPORT': 14, 'NEI': 9}
corpus_fit: {'near': 17, 'far': 13}
OK - claim set is clean


In [7]:
#running the case study (the long cell)

'''
Encodes the SciFact-Open corpus once, then runs no retrieval, plain dense, and dense plus rerank
at k = 1, 3, 5, 10.
'''

!python realworld/seafood_claims.py \
  --claims_csv realworld/seafood_claims.csv \
  --model1_path models/saved_models/baseline_scifact \
  --model2_path models/saved_models/evidence_scifact \
  --out_dir results/step10_realworld \
  --k_values 1 3 5 10

Device: cuda
Loaded 30 seafood/sustainability claims
Loading SciFact-Open corpus (this is the same corpus as the earlier steps)...
[load_scifact_open] 279 claims loaded (15 had conflicting SUPPORT/CONTRADICT evidence, resolved to SUPPORT).
Corpus: 500000 documents
Loading Model 1 (claim-only) and Model 2 (claim+evidence)...
Loading weights: 100% 201/201 [00:00<00:00, 5986.76it/s]
Loading weights: 100% 201/201 [00:00<00:00, 5914.81it/s]
Building dense retriever (one-time corpus encoding, ~40 min on GPU)...
Loading dense retrieval model: sentence-transformers/all-mpnet-base-v2
Using device: cuda
modules.json: 100% 349/349 [00:00<00:00, 1.55MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 708kB/s]
README.md: 100% 11.6k/11.6k [00:00<00:00, 27.8MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 91.1kB/s]
config.json: 100% 571/571 [00:00<00:00, 3.81MB/s]

model.safetensors: downloading bytes:  35% 155M/438M [00:01<00:01, 149MB/s, 10.3MB/s  ] 
model.safetensors: 

In [8]:
#reading the summary (all three conditions and the H4 anchor comparison)

'''
Accuracy for no retrieval, plain dense, and dense plus rerank at each depth, plus the primary H4
comparison at the anchor depth k = 3 and the exploratory best-across-depths figures. Accuracy over
these 30 claims is an orienting count, not an estimate of real social-media performance.
'''

import json
s = json.load(open('results/step10_realworld/step10_summary.json'))

print("No retrieval:", s['no_retrieval']['overall'], "| by fit:", s['no_retrieval']['by_corpus_fit'])
print()
for k in s['provenance']['k_values']:
    d = s['dense_by_k'][str(k)]['overall']
    r = s['dense_reranked_by_k'][str(k)]['overall']
    print(f"k={k:<2} dense: {d}  |  dense+rerank: {r}")
print()
rv = s['retrieval_vs_no_retrieval']
print(f"PRIMARY H4 at anchor k={rv['anchor_k']}:")
print(f"  no-retrieval {rv['no_retrieval_accuracy_pct']}%  vs  "
      f"dense {rv['dense_accuracy_at_anchor_k']}% (beats? {rv['dense_beats_no_retrieval_at_anchor_k']})  "
      f"vs  dense+rerank {rv['reranked_accuracy_at_anchor_k']}% (beats? {rv['reranked_beats_no_retrieval_at_anchor_k']})")
print(f"Exploratory (best across all k): dense {rv['highest_observed_dense_accuracy_across_tested_k']}%, "
      f"rerank {rv['highest_observed_reranked_accuracy_across_tested_k']}%")

No retrieval: {'n': 30, 'correct': 7, 'accuracy_pct': 23.3} | by fit: {'near': {'n': 17, 'correct': 4, 'accuracy_pct': 23.5}, 'far': {'n': 13, 'correct': 3, 'accuracy_pct': 23.1}}

k=1  dense: {'n': 30, 'correct': 10, 'accuracy_pct': 33.3}  |  dense+rerank: {'n': 30, 'correct': 8, 'accuracy_pct': 26.7}
k=3  dense: {'n': 30, 'correct': 9, 'accuracy_pct': 30.0}  |  dense+rerank: {'n': 30, 'correct': 11, 'accuracy_pct': 36.7}
k=5  dense: {'n': 30, 'correct': 13, 'accuracy_pct': 43.3}  |  dense+rerank: {'n': 30, 'correct': 11, 'accuracy_pct': 36.7}
k=10 dense: {'n': 30, 'correct': 12, 'accuracy_pct': 40.0}  |  dense+rerank: {'n': 30, 'correct': 13, 'accuracy_pct': 43.3}

PRIMARY H4 at anchor k=3:
  no-retrieval 23.3%  vs  dense 30.0% (beats? True)  vs  dense+rerank 36.7% (beats? True)
Exploratory (best across all k): dense 43.3%, rerank 43.3%


In [9]:
#the CONTRADICT confidence check (H3) and confident-wrong (H5)

'''
For each true label, accuracy and mean max-softmax probability, including the mean when the
prediction was wrong. H3 expects wrong predictions on CONTRADICT claims to arrive at relatively
high confidence, mirroring the inversion found in Step 8. Shown for all three conditions.
'''

import json
s = json.load(open('results/step10_realworld/step10_summary.json'))

def show(name, block):
    print(f"--- {name} ---")
    for label, d in block.items():
        print(f"  {label:<10} n={d['n']:<3} acc={d['accuracy_pct']}%  "
              f"mean_max_softmax={d['mean_max_softmax']}  "
              f"n_wrong={d['n_wrong']}  when_wrong={d['mean_max_softmax_when_wrong']}")

show("no_retrieval", s['no_retrieval']['confidence_by_true_label'])
for k in s['provenance']['k_values']:
    show(f"dense k={k}", s['dense_by_k'][str(k)]['confidence_by_true_label'])
    show(f"dense+rerank k={k}", s['dense_reranked_by_k'][str(k)]['confidence_by_true_label'])

--- no_retrieval ---
  CONTRADICT n=7   acc=0.0%  mean_max_softmax=0.8524  n_wrong=7  when_wrong=0.8524
  NEI        n=9   acc=55.6%  mean_max_softmax=0.7667  n_wrong=4  when_wrong=0.8657
  SUPPORT    n=14  acc=14.3%  mean_max_softmax=0.8312  n_wrong=12  when_wrong=0.8258
--- dense k=1 ---
  CONTRADICT n=7   acc=0.0%  mean_max_softmax=0.9616  n_wrong=7  when_wrong=0.9616
  NEI        n=9   acc=55.6%  mean_max_softmax=0.9094  n_wrong=4  when_wrong=0.8157
  SUPPORT    n=14  acc=35.7%  mean_max_softmax=0.8901  n_wrong=9  when_wrong=0.9183
--- dense+rerank k=1 ---
  CONTRADICT n=7   acc=0.0%  mean_max_softmax=0.964  n_wrong=7  when_wrong=0.964
  NEI        n=9   acc=55.6%  mean_max_softmax=0.873  n_wrong=4  when_wrong=0.7391
  SUPPORT    n=14  acc=21.4%  mean_max_softmax=0.9555  n_wrong=11  when_wrong=0.9605
--- dense k=3 ---
  CONTRADICT n=7   acc=0.0%  mean_max_softmax=0.9083  n_wrong=7  when_wrong=0.9083
  NEI        n=9   acc=22.2%  mean_max_softmax=0.7899  n_wrong=7  when_wrong=0.7327

In [10]:
#inspecting retrieved documents at the anchor depth (for the manual annotation)

'''
For the qualitative analysis, this prints each claim at k = 3 under both retrieval conditions
(dense and dense plus rerank), so relevance can be judged by hand and the reranker's effect read
off by comparing the two. This is the starting point for the annotation table defined in
results_annotation_guide.md.
'''

import json
records = json.load(open('results/step10_realworld/step10_records.json'))

def rows_for(cond):
    return sorted([r for r in records if r.get('condition') == cond and r.get('k') == 3],
                  key=lambda x: x['claim_id'])

for cond in ['dense', 'dense_reranked']:
    print("=" * 70)
    print(f"CONDITION: {cond}  (k=3)")
    print("=" * 70)
    for r in rows_for(cond):
        flag = "OK " if r['correct'] else "XX "
        print(f"{flag}{r['claim_id']} [{r['true_label']}->{r['predicted_label']} "
              f"conf={r['confidence']:.2f}] fit={r.get('corpus_fit')}")
        print(f"     claim: {r['claim'][:88]}")
        print(f"     retrieved: {r.get('retrieved_doc_ids')}")
    print()

CONDITION: dense  (k=3)
XX c01 [CONTRADICT->SUPPORT conf=0.89] fit=near
     claim: Wild salmon has more omega-3 than farmed salmon
     retrieved: ['89404357', '89174559', '433517']
XX c02 [SUPPORT->NEI conf=0.99] fit=far
     claim: MSC certification does not prohibit bottom trawling
     retrieved: ['56110152', '29071163', '39449195']
XX c03 [SUPPORT->NEI conf=0.92] fit=near
     claim: Farmed salmon is safer to eat raw because it has fewer parasites than wild salmon
     retrieved: ['1477180', '6986993', '8847058']
XX c04 [SUPPORT->CONTRADICT conf=0.60] fit=near
     claim: Freezing fish kills parasites
     retrieved: ['1690511', '12099781', '28635872']
OK c05 [SUPPORT->SUPPORT conf=0.96] fit=far
     claim: Farmed salmon commonly has sea lice infestations
     retrieved: ['86637883', '12831028', '2284990']
XX c06 [NEI->CONTRADICT conf=0.70] fit=near
     claim: Cooking fish at high heat destroys most of its omega-3 content
     retrieved: ['433517', '16056267', '7913654']
OK c07 

In [11]:
#saving the Step 10 outputs to Drive
import os, shutil

dst = '/content/drive/MyDrive/rag-thesis/results/step10_realworld'
os.makedirs(dst, exist_ok=True)
src_dir = '/content/retrieval-effects-claim-verification/results/step10_realworld'
saved = []
for f in sorted(os.listdir(src_dir)):
    p = os.path.join(src_dir, f)
    if os.path.isfile(p):
        shutil.copy(p, os.path.join(dst, f))
        saved.append(f)
print("Saved to Drive:", saved)

Saved to Drive: ['step10_records.json', 'step10_summary.json']
